In [0]:
import os
import json
import shutil
import uuid
from pathlib import Path
import pandas as pd

CATALOG = "workspace"
SCHEMA_BRONZE = "Bronze"
SCHEMA_SILVER = "Silver"
SCHEMA_GOLD = "Gold"
SCHEMA_STAGING = "elt_lab"
VOLUME = "elt_volume"

NOTEBOOK_ROOT = "/Shared/elt_lab"

BRONZE_ORDERS_TABLE = f"{CATALOG}.{SCHEMA_BRONZE}.orders"
BRONZE_CUSTOMERS_TABLE = f"{CATALOG}.{SCHEMA_BRONZE}.customers"
REJECT_ORDERS_TABLE = f"{CATALOG}.{SCHEMA_SILVER}.reject_orders"
SILVER_ORDERS_TABLE = f"{CATALOG}.{SCHEMA_SILVER}.orders"
SILVER_CUSTOMERS_TABLE = f"{CATALOG}.{SCHEMA_SILVER}.customers"
GOLD_DAILY_SALES_TABLE = f"{CATALOG}.{SCHEMA_GOLD}.daily_sales"
GOLD_SALES_BY_COUNTRY_TABLE = f"{CATALOG}.{SCHEMA_GOLD}.sales_by_country"
OPS_PIPELINE_EVENTS = f"{CATALOG}.{SCHEMA_STAGING}.ops_pipeline_events"
OPS_PIPELINE_ALERTS = f"{CATALOG}.{SCHEMA_STAGING}.ops_pipeline_alerts"
OPS_DATASET_VERSIONS = f"{CATALOG}.{SCHEMA_STAGING}.ops_dataset_versions"

VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA_STAGING}/{VOLUME}"

LANDING_ROOT = f"{VOLUME_ROOT}/landing"
LANDING_ORDERS_ROOT = f"{LANDING_ROOT}/orders"
LANDING_CUSTOMERS_ROOT = f"{LANDING_ROOT}/customers"

WORK_ROOT = f"{VOLUME_ROOT}/work"
BRONZE_WORK_ROOT = f"{WORK_ROOT}/bronze"
BRONZE_WORK_ORDERS_ROOT = f"{BRONZE_WORK_ROOT}/orders"
BRONZE_WORK_CUSTOMERS_ROOT = f"{BRONZE_WORK_ROOT}/customers"

SILVER_WORK_ROOT = f"{WORK_ROOT}/silver"
SILVER_CURRENT_ROOT = f"{SILVER_WORK_ROOT}/current"

GOLD_WORK_ROOT = f"{WORK_ROOT}/gold"
GOLD_CURRENT_ROOT = f"{GOLD_WORK_ROOT}/current"

MONITORING_ROOT = f"{VOLUME_ROOT}/monitoring"

REJECT_THRESHOLD = 0.07 # 0.05

print("--- TABLAS ---")
print(f"BRONZE_ORDERS_TABLE:         {BRONZE_ORDERS_TABLE}")
print(f"BRONZE_CUSTOMERS_TABLE:      {BRONZE_CUSTOMERS_TABLE}")
print(f"REJECT_ORDERS_TABLE:         {REJECT_ORDERS_TABLE}")
print(f"SILVER_ORDERS_TABLE:         {SILVER_ORDERS_TABLE}")
print(f"SILVER_CUSTOMERS_TABLE:      {SILVER_CUSTOMERS_TABLE}")
print(f"GOLD_DAILY_SALES_TABLE:      {GOLD_DAILY_SALES_TABLE}")
print(f"GOLD_SALES_BY_COUNTRY_TABLE: {GOLD_SALES_BY_COUNTRY_TABLE}")
print(f"OPS_PIPELINE_EVENTS:         {OPS_PIPELINE_EVENTS}")
print(f"OPS_PIPELINE_ALERTS:         {OPS_PIPELINE_ALERTS}")
print(f"OPS_DATASET_VERSIONS:        {OPS_DATASET_VERSIONS}")

print("\n--- RUTAS ---")
print(f"NOTEBOOK_ROOT:               {NOTEBOOK_ROOT}")
print(f"VOLUME_ROOT:                 {VOLUME_ROOT}")
print(f"LANDING_ROOT:                {LANDING_ROOT}")
print(f"LANDING_ORDERS_ROOT:         {LANDING_ORDERS_ROOT}")
print(f"LANDING_CUSTOMERS_ROOT:      {LANDING_CUSTOMERS_ROOT}")
print(f"WORK_ROOT:                   {WORK_ROOT}")
print(f"BRONZE_WORK_ROOT:            {BRONZE_WORK_ROOT}")
print(f"BRONZE_WORK_ORDERS_ROOT:     {BRONZE_WORK_ORDERS_ROOT}")
print(f"BRONZE_WORK_CUSTOMERS_ROOT:  {BRONZE_WORK_CUSTOMERS_ROOT}")
print(f"SILVER_WORK_ROOT:            {SILVER_WORK_ROOT}")
print(f"SILVER_CURRENT_ROOT:         {SILVER_CURRENT_ROOT}")
print(f"GOLD_WORK_ROOT:              {GOLD_WORK_ROOT}")
print(f"GOLD_CURRENT_ROOT:           {GOLD_CURRENT_ROOT}")
print(f"MONITORING_ROOT:             {MONITORING_ROOT}")

print("\n--- CONFIGURACIÓN ---")
print(f"REJECT_THRESHOLD:            {REJECT_THRESHOLD}")

In [0]:
def get_text_widget(name: str, default: str = "") -> str:
    try:
        dbutils.widgets.text(name, default)
    except Exception:
        pass
    return dbutils.widgets.get(name)

def get_or_create_run_id() -> str:
    run_id = get_text_widget("run_id", "").strip()
    return run_id if run_id else str(uuid.uuid4())

def ensure_dir(path: str) -> None:
    # Crea el directorio y sus padres si no existen
    # En dbutils, si la ruta ya es 'file:/...', asegúrate de limpiarla o usarla directamente.
    # Los paths de Volumes generalmente empiezan con /Volumes/
    dbutils.fs.mkdirs(path)

def clear_dir(path: str) -> None:
    # Intenta eliminar el directorio y todo su contenido (recurse=True)
    try:
        dbutils.fs.rm(path, recurse=True)
    except Exception as e:
        # Si la carpeta no existía previamente, dbutils.fs.rm puede arrojar error, lo ignoramos
        pass
    
    # Vuelve a crear el directorio limpio
    dbutils.fs.mkdirs(path)

def assert_file_exists(path: str) -> str:
    if not os.path.exists(path):
        raise FileNotFoundError(f"No existe el archivo: {path}")
    return path

def list_files(root: str, pattern: str):
    root_path = Path(root)
    if not root_path.exists():
        return []
    return sorted(str(p) for p in root_path.rglob(pattern) if p.is_file())

def read_all_csv(root: str, columns=None) -> pd.DataFrame:
    files = list_files(root, "*.csv")
    if not files:
        return pd.DataFrame(columns=columns or [])
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

def safe_pdf(pdf: pd.DataFrame) -> pd.DataFrame:
    out = pdf.copy()
    return out.where(pd.notnull(out), None)

def json_str(obj):
    if obj is None:
        return None
    return json.dumps(obj, ensure_ascii=False, default=str)

def utc_now_naive():
    return pd.Timestamp.utcnow().to_pydatetime().replace(tzinfo=None)

def write_csv(pdf: pd.DataFrame, path: str) -> None:
    ensure_dir(str(Path(path).parent))
    pdf.to_csv(path, index=False)

def write_jsonl(pdf: pd.DataFrame, path: str) -> None:
    ensure_dir(str(Path(path).parent))
    pdf.to_json(path, orient="records", lines=True, force_ascii=False)

def landing_orders_file(run_date: str) -> str:
    return f"{LANDING_ORDERS_ROOT}/{run_date}/orders.csv"

def landing_customers_file(run_date: str) -> str:
    return f"{LANDING_CUSTOMERS_ROOT}/{run_date}/customers.jsonl"

def bronze_orders_snapshot_file(run_date: str) -> str:
    return f"{BRONZE_WORK_ORDERS_ROOT}/{run_date}/bronze_orders.csv"

def bronze_customers_snapshot_file(run_date: str) -> str:
    return f"{BRONZE_WORK_CUSTOMERS_ROOT}/{run_date}/bronze_customers.csv"

def silver_orders_current_file() -> str:
    return f"{SILVER_CURRENT_ROOT}/silver_orders.csv"

def silver_customers_current_file() -> str:
    return f"{SILVER_CURRENT_ROOT}/silver_customers.csv"

def gold_daily_current_file() -> str:
    return f"{GOLD_CURRENT_ROOT}/gold_daily_sales.csv"

def gold_country_current_file() -> str:
    return f"{GOLD_CURRENT_ROOT}/gold_sales_by_country.csv"